In [ ]:
from google.colab import runtime
runtime.unassign()

# Initialisation (always run this)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ! pip install numpyro
# # ! pip install jax_cosmo
# ! pip install arviz
! pip install jaxopt
! pip install corner
! pip install emcee

path = '/content/drive/MyDrive/SchwarMAX-MCMC/'

import sys
sys.path.append(path)

from model import *
from likelihoods import *
from utils import *
from sample_from_density import sample_from_density_grid
from CylindricalSpline import get_phi_m, evaluate_phi_axisymmetric

import os
os.environ["JAX_ENABLE_X64"] = "True"

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import jax.numpy.linalg as jnn
import pandas as pd
import numpy as np
import scipy as sp
import pickle

import emcee
import corner
import matplotlib.pyplot as plt

from constants import EPSILON

def get_dict_data(path):

    with open(path + 'mock_Nbody_disc_bulge_XY_withRot.pkl', 'rb') as f:
        bin_dict = pickle.load(f)

    # voronoi binning mapping and data
    num_per_bin = jnp.array(bin_dict['num_per_bin'])
    total_bins = jnp.array(bin_dict['total_bins'])
    bin_mapping = jnp.array(bin_dict['bin_mapping'])
    surface_density = jnp.array(bin_dict['surface_density'])
    V_data = jnp.array(bin_dict['V_mean'])
    sigma_data = jnp.array(bin_dict['V_sigma'])
    h1_data = jnp.array(bin_dict['h1'])
    h2_data = jnp.array(bin_dict['h2'])
    h3_data = jnp.array(bin_dict['h3'])
    h4_data = jnp.array(bin_dict['h4'])
    v0 = jnp.array(bin_dict['v0'])
    s = jnp.array(bin_dict['s'])
    alpha, beta, gamma = bin_dict['orientation']

    # V_data_err = jnp.where(0.1 * jnp.fabs(V_data) < 10, 10, 0.1 * V_data)
    # sigma_data_err = jnp.where(0.1 * jnp.fabs(sigma_data) < 5, 5, 0.1 * sigma_data)
    # h1_data_err = jnp.where(0.1 * jnp.fabs(h1_data) < 0.03, 0.03, 0.1 * jnp.fabs(h1_data))
    # h2_data_err = jnp.where(0.1 * jnp.fabs(h2_data) < 0.03, 0.03, 0.1 * jnp.fabs(h2_data))
    # h3_data_err = jnp.where(0.1 * jnp.fabs(h3_data) < 0.03, 0.03, 0.1 * jnp.fabs(h3_data))
    # h4_data_err = jnp.where(0.1 * jnp.fabs(h4_data) < 0.03, 0.03, 0.1 * jnp.fabs(h4_data))
    V_data_err = jnp.array(bin_dict['V_mean_err'])
    sigma_data_err = jnp.array(bin_dict['V_sigma_err'])
    h1_data_err = jnp.array(bin_dict['h1_err'])
    h2_data_err = jnp.array(bin_dict['h2_err'])
    h3_data_err = jnp.array(bin_dict['h3_err'])
    h4_data_err = jnp.array(bin_dict['h4_err'])

    # df_Rzphi_data = pd.read_csv(path + 'mock_axisymmetric_disc_Rzphi.csv')
    # Rzphi_density_data = jnp.array(df_Rzphi_data['mass'].to_numpy()).astype(jnp.float32)
    with open(path + 'mock_axisymmetric_disc_Rzphi.pkl', 'rb') as f:
        Rzphi_density_data = pickle.load(f)

    R_grid, z_grid, phi_grid = Rzphi_density_data['R_grid'], Rzphi_density_data['z_grid'], Rzphi_density_data['phi_grid']
    dR = np.unique(R_grid)[1] - np.unique(R_grid)[0]
    dz = np.unique(z_grid)[1] - np.unique(z_grid)[0]
    dphi = np.unique(phi_grid)[1] - np.unique(phi_grid)[0]
    sample_for_integration = Rzphi_density_data['sample_for_integration']

    from scipy.stats import qmc
    X_regular_grid, Y_regular_grid = bin_dict['X_regular_grid'], bin_dict['Y_regular_grid']
    dX = jnp.unique(X_regular_grid)[1] - jnp.unique(X_regular_grid)[0]
    dY = jnp.unique(Y_regular_grid)[1] - jnp.unique(Y_regular_grid)[0]
    sampler = qmc.Sobol(d=3, scramble=False)
    sample = sampler.random_base2(m=10)


    dict_data = {
        # 'w0': w0,
        'v0': v0,
        's': s,

        # 'Rzphi_density_data': Rzphi_density_data,
        'XY_density_data': surface_density,
        'V_data': V_data,
        'V_data_err': V_data_err,
        'sigma_data': sigma_data,
        'sigma_data_err': sigma_data_err,
        'h1_data': h1_data,
        'h1_data_err': h1_data_err,
        'h2_data': h2_data,
        'h2_data_err': h2_data_err,
        'h3_data': h3_data,
        'h3_data_err': h3_data_err,
        'h4_data': h4_data,
        'h4_data_err': h4_data_err,
        'num_per_bin': num_per_bin,
        'bin_mapping': bin_mapping,
        'total_bins': total_bins.item(),

        'R_grid': R_grid,
        'z_grid': z_grid,
        'phi_grid': phi_grid,
        'dR': dR,
        'dz': dz,
        'dphi': dphi,
        'sample_for_integration': sample_for_integration,

        'X_regular_grid': X_regular_grid,
        'Y_regular_grid': Y_regular_grid,
        'dX': dX,
        'dY': dY,
        'sample_for_integration_XY': sample,
    }

    return dict_data

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.4/172.4 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 4.9 MB/s eta 0:00:00


# Load data and preprocesses (always run this)

In [ ]:
path = '/content/drive/MyDrive/SchwarMAX-MCMC/'
dict_data = get_dict_data(path)

def log_prior(theta,):
    if (6 < theta[0] < 10) and (8 < theta[1] < 12) and (-1 < theta[2] < 2) and (-1 < theta[3] < 1) and (-1 < theta[4] < 1)\
    and (0 <= theta[5] < jnp.pi) and (0 <= theta[6] < jnp.pi/2) and (0 <= theta[7] < jnp.pi):
        return 0.0  # log(1) = 0 for uniform prior
    return -np.inf  # log(0) = -inf for out-of-bounds

def log_prob(theta,):
    # print(theta)
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf

    ll = logl_density(theta, dict_data, dict_data['total_bins'])

    return ll + lp

ndim = 8
nwalkers = 16  # must be >= 2 * ndim

# Initialize walkers around ground truth
# p0 = np.array([ground_truth[k] for k in param_names])
p0 = np.array([9.2, 10, 0.3, 0., 0., jnp.pi/4, jnp.pi/4, jnp.pi/4])
# initial_pos = p0 + 1e-1 * np.random.randn(nwalkers, ndim)
np.random.seed(42)
initial_pos = p0 + np.random.uniform(-0.3, 0.3, (nwalkers, ndim))

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)
sampler.run_mcmc(initial_pos, 500, progress=True)

samples = sampler.get_chain(discard=200, flat=True)


params_bestfit = np.percentile(samples, axis=0, q=50)
logl_val = logl_density(params_bestfit, dict_data, dict_data['total_bins'])
print('Best-fit logL projection', logl_val)#

dict_data['logl_density_max'] = logl_val

logrho0_best_fit, logM_bulge_best_fit, \
logRd_disc_best_fit, logHs_disc_best_fit, logRs_bulge_best_fit, \
alpha_best_fit, beta_best_fit, gamma_best_fit = params_bestfit
# logMhalo_best_fit, logrho0_best_fit, logM_bulge_best_fit, logRh_disk_best_fit, logRs_disk_best_fit, logHs_disk_best_fit, logRs_bulge_best_fit,\
#       alpha_best_fit, beta_best_fit, gamma_best_fit, logLM_best_fit = (11.8, 8.8, 10.4, 1.2, 0.45, -0.24, -0.1, 30*np.pi/180, 20*np.pi/180, 0*np.pi/180, 0)

print('logrho0_best_fit',logrho0_best_fit)
print('logM_bulge_best_fit',logM_bulge_best_fit)
print('logRd_disc_best_fit',logRd_disc_best_fit)
print('logHs_disk_best_fit',logHs_disc_best_fit)
print('logRs_bulge_best_fit',logRs_bulge_best_fit)
print('alpha_best_fit',alpha_best_fit * 180 / np.pi)
print('beta_best_fit',beta_best_fit * 180 / np.pi)
print('gamma_best_fit',gamma_best_fit * 180 / np.pi)

params_halo_pot = {
    'logM': 11.8,
    'Rs':19,
    'a':1.0,
    'b':1.0,
    'c':1.0,
    'x_origin':0.0,
    'y_origin':0.0,
    'z_origin':0.0,
    'dirx':0.0,
    'diry':0.0,
    'dirz':1.0
}

params_disk_rho = {
    'rho0_disc': 10 ** logrho0_best_fit,
    'Rd_disc': 10 ** logRd_disc_best_fit,
    'hz_disc': 10 ** logHs_disc_best_fit,
    'x_origin': 0.0,
    'y_origin': 0.0,
    'z_origin': 0.0,
    'dirx': 0.0,
    'diry': 0.0,
    'dirz': 1.0,
    'alpha': alpha_best_fit * 180 / jnp.pi,
    'beta': beta_best_fit * 180 / jnp.pi,
    'gamma': gamma_best_fit * 180 / jnp.pi,
    'light_to_mass_ratio': 1,
    'logM_bulge': logM_bulge_best_fit,
    'Rs_bulge': 10 ** logRs_bulge_best_fit,
}

@jax.jit
def potential_func(x, y, z, dict_phi, params_halo):
    """ Returns Phi(R, z) """
    phi_halo = NFW_potential(x, y, z, params_halo)
    phi_disk = evaluate_phi_axisymmetric(x, y, z, dict_phi)
    return phi_halo + phi_disk

@jax.jit
def density_func(x, y, z, params):
    """ Returns Stellar Density nu(R, z) """
    # Double Exponential Disk
    val = DoubleExponentialDisk_density(x, y, z, params) + Hernquist_density(x, y, z, params)
    return val

bounds = jnp.array(
    [
        [-15.0, 15.0],  # x
        [-15.0, 15.0],  # y
        [-5.0, 5.0],    # z
    ],
    dtype=jnp.float32,
)

n_samples = 20_000
n_x,n_y,n_z = 48, 48, 32

key = jax.random.PRNGKey(0)
sample_ic_dict = sample_from_density_grid(
    key,
    density_func,
    params_disk_rho,
    bounds,
    n_samples=n_samples,
    n_x=n_x,
    n_y=n_y,
    n_z=n_z,
)
samples = np.asarray(sample_ic_dict["samples"])
dict_data['w0'] = samples

100%|██████████| 500/500 [00:18<00:00, 27.46it/s]


Best-fit logL projection -2.2253633829428727
logrho0_best_fit 9.155161161178649
logM_bulge_best_fit 9.793966499163814
logRd_disc_best_fit 0.34465156073228176
logHs_disk_best_fit -0.2190522488126071
logRs_bulge_best_fit -0.11881893273369626
alpha_best_fit 29.539100153736705
beta_best_fit 21.886298064938483
gamma_best_fit 44.90103555130126


# MCMC

In [ ]:
ground_truth = [
    11.5,
    logrho0_best_fit,
    logM_bulge_best_fit,
    jnp.log10(19).item(),
    logRd_disc_best_fit,
    logHs_disc_best_fit,
    logRs_bulge_best_fit,
    alpha_best_fit,
    beta_best_fit,
    gamma_best_fit,
    -0.3
]
logL = logl_angular_input(ground_truth, dict_data, dict_data['total_bins'])
print(logL)

import time
start = time.time()
logL = logl_angular_input(ground_truth, dict_data, dict_data['total_bins'])
logL.block_until_ready()  # Ensure computation finishes before timing
end = time.time()
print('time per logl evaluation', end - start, 's')
print(logL)

-439.13200636271046
time per logl evaluation 19.49850296974182 s
-440.30219323331886


In [ ]:
path

'/content/drive/MyDrive/SchwarMAX-MCMC/'

In [ ]:
ground_truth = [
    12.0,
    logrho0_best_fit,
    logM_bulge_best_fit,
    jnp.log10(19).item(),
    logRd_disc_best_fit,
    logHs_disc_best_fit,
    logRs_bulge_best_fit,
    alpha_best_fit,
    beta_best_fit,
    gamma_best_fit,
    -0.
]
logL = logl_angular_input(ground_truth, dict_data, dict_data['total_bins'])
print(logL)

import time
start = time.time()
logL = logl_angular_input(ground_truth, dict_data, dict_data['total_bins'])
logL.block_until_ready()  # Ensure computation finishes before timing
end = time.time()
print('time per logl evaluation', end - start)
print(logL)

prior_uniform_low =  [
    ground_truth[0] - 3,
    ground_truth[1] - 3,
    ground_truth[2] - 3,
    ground_truth[3] - 1,
    ground_truth[4] - 1,
    ground_truth[5] - 1,
    ground_truth[6] - 1,
    0,
    0,
    0,
    -2
]
prior_uniform_high = [
    ground_truth[0] + 3,
    ground_truth[1] + 3,
    ground_truth[2] + 3,
    ground_truth[3] + 1,
    ground_truth[4] + 1,
    ground_truth[5] + 1,
    ground_truth[6] + 1,
    jnp.pi,
    jnp.pi/2,
    jnp.pi,
    2
]

def log_prior(params):
    lp = 0
    for i in range (0, ndim):
        if (params[i]<=prior_uniform_low[i]) & (params[i]>=prior_uniform_high[i]):
            lp+= -jnp.inf
    return lp

def log_prob(theta):
    params = theta
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf
    ll = float(logl_angular_input(params, dict_data, dict_data['total_bins']))  # convert from JAX array
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll

ndim = 11
nwalkers = 22  # must be >= 2 * ndim

np.random.seed(42)

# Initialize walkers around ground truth
# p0 = ground_truth
# initial_pos = np.zeros((nwalkers, ndim))
# initial_pos[:, :7] = np.vstack([p0[:7]]*nwalkers) * np.ones((nwalkers, 7)) + np.random.uniform(-0.5, 0.5, (nwalkers, 7))
# initial_pos[:,7:10] = np.vstack([p0[7:10]]*nwalkers) * np.ones((nwalkers, 3)) + np.random.uniform(-0.1, 0.1, (nwalkers, 3))
# initial_pos[:,7:10] = np.clip(initial_pos[:,7:10], a_min=0, a_max=np.vstack([[jnp.pi, jnp.pi/2, jnp.pi]]*nwalkers))
# initial_pos[:,  10] = p0[10] + np.random.uniform(-0.5, 0.5, nwalkers)

# backend = emcee.backends.HDFBackend(path+'/backend_0228.csv')
# backend.reset(nwalkers, ndim)
# sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob, backend=backend)
# sampler.run_mcmc(initial_pos, 250, progress=True)

# Restart the sampler from the backend
backend = emcee.backends.HDFBackend(path+'/backend_0228.csv')
sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob, backend=backend)
state = backend.get_last_sample()   # this is an emcee.State
print('restarted from the state:', state)
sampler.run_mcmc(state, 100, progress=True)   # adds 500 more steps

samples = sampler.get_chain(discard=80, flat=True)

param_names = ['logM_halo','logM_disk','logM_bulge', 'logRs_halo', 'logRs_disk', 'logHs_disk', 'logRs_bulge', 'alpha', 'beta', 'gamma', 'log_light_to_mass_ratio']
import pandas as pd
pd.DataFrame(samples, columns=param_names).to_csv(path+'/test_posterior_0228.csv', index=False)

samples_raw = sampler.get_chain(discard=0, flat=False)
log_prob_raw = sampler.get_log_prob(discard=0, flat=False)
with open(path+'/test_posterior_WholeChain_0228.pkl', 'wb') as f:
    pickle.dump(samples_raw, f)
with open(path+'/test_posterior_logprob_0228.pkl', 'wb') as f:
    pickle.dump(log_prob_raw, f)

-65.36249871222125
time per logl evaluation 27.569215059280396
-63.06812992963805
restarted from the state: State([[11.82253432  9.45002665  9.68443302  1.4272352   0.31357096 -0.25173511
  -0.14951871  0.50171154  0.36212648  0.80823272  0.03350621]
 [12.58179902  9.40531558  9.5481564   1.8876777   0.31480232 -0.20759498
  -1.28280168  0.53087731  0.33554183  0.87714366 -0.08584919]
 [11.81197649  9.43299498  9.5148207   1.70749211  0.32091345 -0.22399454
  -1.03019884  0.54807696  0.32554378  0.81742493  0.14205967]
 [11.97759759  9.33144031  9.44847291  1.23771005  0.28603585 -0.24960727
  -0.09879437  0.46452206  0.37242385  0.82389858  0.14961482]
 [11.388061    9.50323997  9.93266978  1.22866192  0.29904991 -0.24520679
   0.68331526  0.47873201  0.34981145  0.69120255 -0.05073818]
 [11.98087918  9.37467694  9.45289516  1.30085415  0.25807539 -0.21532304
  -0.12502552  0.55503299  0.34171557  0.71982715  0.14947621]
 [11.72638009  9.41426477  9.64952012  1.20669498  0.30117003 -0

100%|██████████| 100/100 [15:16:10<00:00, 549.70s/it]


# Grid search

In [ ]:
def log_prior(params):
    lp = 0
    for i in range (0, ndim):
        if (params[i]<=prior_uniform_low[i]) & (params[i]>=prior_uniform_high[i]):
            lp+= -jnp.inf
    return lp

def log_prob(theta):
    params = theta
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf
    ll = float(logl_angular_input(params, dict_data, dict_data['total_bins']))  # convert from JAX array
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll

ground_truth = [
    11.5,
    logrho0_best_fit,
    logM_bulge_best_fit,
    jnp.log10(19).item(),
    logRd_disc_best_fit,
    logHs_disc_best_fit,
    logRs_bulge_best_fit,
    alpha_best_fit,
    beta_best_fit,
    gamma_best_fit,
    0.
]
prior_uniform_low =  [
    ground_truth[0] - 3,
    ground_truth[1] - 3,
    ground_truth[2] - 3,
    ground_truth[3]- 1,
    ground_truth[4]- 1,
    ground_truth[5]- 1,
    ground_truth[6]- 1,
    0,
    0,
    0,
    -2
]
prior_uniform_high = [
    ground_truth[0] + 3,
    ground_truth[1] + 3,
    ground_truth[2] + 3,
    ground_truth[3]+ 1,
    ground_truth[4]+ 1,
    ground_truth[5]+ 1,
    ground_truth[6]+ 1,
    jnp.pi,
    jnp.pi/2,
    jnp.pi,
    2
]

ndim = 11

n_grid = 1024
param_grid = pd.read_csv(path + '/quasi_random_samples_12D_unity.csv').to_numpy()
index = np.random.choice(len(param_grid), size=n_grid, replace=False)
param_grid = param_grid[index]
param_grid[:, 0] = (param_grid[:, 0] - 0.5) * 6 + ground_truth[0]
param_grid[:, 1] = (param_grid[:, 1] - 0.5) * 6 + ground_truth[1]
param_grid[:, 2] = (param_grid[:, 2] - 0.5) * 6 + ground_truth[2]
param_grid[:, 2] = (param_grid[:, 3] - 0.5) * 2 + ground_truth[3]
param_grid[:, 3] = (param_grid[:, 4] - 0.5) * 2 + ground_truth[4]
param_grid[:, 4] = (param_grid[:, 5] - 0.5) * 2 + ground_truth[5]
param_grid[:, 4] = (param_grid[:, 6] - 0.5) * 2 + ground_truth[6]
param_grid[:, 5] = (param_grid[:, 7] - 0.5) * 0.1 * jnp.pi + ground_truth[7]
param_grid[:, 6] = (param_grid[:, 8] - 0.5) * 0.1 * jnp.pi + ground_truth[8]
param_grid[:, 7] = (param_grid[:, 9] - 0.5) * 0.1 * jnp.pi + ground_truth[9]
param_grid[:, 8] = (param_grid[:, 10] - 0.5) * 2

from tqdm import tqdm
log_prob_grid = []
for i in tqdm(range(n_grid)):
  log_prob_grid.append(log_prob(param_grid[i]))
log_prob_grid = np.array(log_prob_grid)

pd.DataFrame({
    'logM_halo': param_grid[:, 0],
    'logM_disk': param_grid[:, 1],
    'logM_bulge': param_grid[:, 2],
    'logRs_halo': param_grid[:, 3],
    'logRs_disk': param_grid[:, 4],
    'logHs_disk': param_grid[:, 5],
    'logRs_bulge': param_grid[:, 6],
    'alpha': param_grid[:, 7],
    'beta': param_grid[:, 8],
    'gamma': param_grid[:, 9],
    'log_light_to_mass_ratio': param_grid[:, 10],
    'log_prob': log_prob_grid,
}).to_csv(path + '/grid_search_result_0225.csv', index=False)